# step B — RQ2 관측 (A2 생성): 어텐션·‖v‖·‖av‖

**대응 RQ:** RQ2 — 준수 실패를 매개하는 신호가 코드의 **형태**인가, 그리고 그 신호가 내부 어디에 있는가(관측). 인과는 step C.

**무엇을 확인하나**
- 선행에 camel/snake 이름을 **6/6 균형 배치**(이름 짝 풀 80개에서 seed로 12개).
- 새 함수 **이름을 생성하는 바로 그 시점**의 query가 두 표기 그룹을 각각 얼마나 보는지(축 A),
  각 그룹의 ‖v‖(축 B 크기)·‖av‖(축 A×B)를 층별로 잰다.
- 기대: **지침 어텐션은 조건 간 평탄** + **충돌 표기 토큰을 더 봄**(camel 지침→snake 토큰↑, snake 지침→camel 토큰↑).

설계 문서: `docs/stepB/plan.md` (표본 설계·GQA 방법 A).

> **메모리(무료 T4):** Qwen2.5-Coder-3B-Instruct fp16 ≈ 6.2GB. 관측은 `output_attentions`로 전체 어텐션을 뜨지만
> stepB 합성 프롬프트는 수백 토큰이라 T4에서 감당된다(§2.7의 15GB/층은 16K 컨텍스트 얘기).
> 어텐션 가중치를 읽으려면 **eager 어텐션**으로 로드한다(`attn_implementation='eager'`).
> **재개 가능:** 조건마다 즉시 저장하고, 이미 저장된 조건은 로드만 한다. 끊겨도 셀 4·5·6을 다시 실행하면 이어서 한다.

In [ ]:
# 환경 설정 — 설치, GPU 확인, 시드 고정
!pip install -q transformers accelerate torch matplotlib pandas

import random, numpy as np, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (주의: 매우 느림)')

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('seed fixed:', SEED)

In [ ]:
# 저장소 클론 및 브랜치 체크아웃
import os
if not os.path.isdir('HCLT_2026'):
    !git clone https://github.com/deanjs/HCLT_2026.git
%cd HCLT_2026
!git fetch --quiet origin stepB/attention-observe
!git checkout stepB/attention-observe
!git pull --quiet origin stepB/attention-observe
!pip install -e . -q
import sys; sys.path.insert(0, 'src')

In [ ]:
# 조건 설정 — 이 실험이 쓰는 조건 축 값
from harness.conditions import (Condition, ModelSpec, PrecedingCode, Instruction,
                                Composition, InstructionForm, Notation)

MODEL = ModelSpec(name='Qwen/Qwen2.5-Coder-3B-Instruct', family='qwen', dtype='float16')
TARGETS = [Notation.CAMEL, Notation.SNAKE]   # 지침 2종 (선행은 항상 camel6/snake6)
SEEDS = list(range(20))                       # 조건당 반복 (seed마다 다른 12개 이름)
REF_FRAC = 0.7                                # 누적/요약에 쓸 기준 층의 상대 위치

def make(target, s):
    return Condition(
        model=MODEL,
        preceding=PrecedingCode(n_compliant=6, n_functions=12, composition=Composition.POOL),
        instruction=Instruction(form=InstructionForm.POSITIVE, target_notation=target),
        seed=s,
    )

conditions = [make(t, s) for t in TARGETS for s in SEEDS]

# 실행 전 예측 (결과와 함께 보존, CLAUDE.md §6)
PREDICTION = ('지침 구간 어텐션은 조건 간 평탄. 충돌 표기 토큰을 더 봄 '
              '(camel 지침->snake 토큰, snake 지침->camel 토큰). ‖v‖ 크기는 평탄.')
print(len(conditions), '개 조건 =', len(TARGETS), 'x', len(SEEDS))

In [ ]:
# 실행 — 조건별 관측 + 즉시 저장(재개 가능). 로직은 harness가 수행.
# 중간중간 '지금까지 누적된' camel vs snake 어텐션을 기준 층에서 요약 출력한다.
from collections import defaultdict
from harness import run, ResultRecord, save_result, result_path
from harness.results import load_result
from harness.model import load_model

handle = load_model(MODEL, attn_implementation='eager')   # 어텐션 가중치 읽기 위해 eager
REF_LAYER = int(handle.num_layers * REF_FRAC)
print('layers:', handle.num_layers, '| GQA:', handle.gqa_info(), '| 기준층 L%d' % REF_LAYER)

acc = defaultdict(lambda: {'camel': [], 'snake': []})   # 지침별 (camel토큰, snake토큰) 어텐션 누적
new = skipped = 0
for i, c in enumerate(conditions, 1):
    p = result_path(c, step='stepB')
    if p.exists():                                       # 이미 한 조건 → 로드(재개)
        rec = load_result(p); skipped += 1
    else:
        out = run(c, handle=handle, mode='observe')
        save_result(ResultRecord(condition=out.condition, metrics=out.metrics,
                                 step='stepB', rq='RQ2', prediction=PREDICTION))
        rec = load_result(p); new += 1
    # 기준 층 누적
    pl = rec.metrics.per_layer.get(REF_LAYER, {})
    tgt = rec.condition.instruction.target_notation.value
    if 'code_camel__attention_weight' in pl and 'code_snake__attention_weight' in pl:
        acc[tgt]['camel'].append(pl['code_camel__attention_weight'])
        acc[tgt]['snake'].append(pl['code_snake__attention_weight'])
    if i % 10 == 0 or i == len(conditions):              # 중간 누적 출력
        print(f'[{i}/{len(conditions)}] 새 {new} / 건너뜀 {skipped}  (기준 L{REF_LAYER})')
        for t in ('camel', 'snake'):
            cs, ss = acc[t]['camel'], acc[t]['snake']
            if cs:
                mc, ms = sum(cs)/len(cs), sum(ss)/len(ss)
                more = 'snake' if ms > mc else 'camel'
                print(f'    지침={t}: camel토큰 {mc:.4f} vs snake토큰 {ms:.4f} '
                      f'-> {more} 더 봄 (n={len(cs)})')
print(f'완료: 새로 {new}, 건너뜀 {skipped}, 총 {len(conditions)}')

In [ ]:
# 결과 로드 — results/stepB/ 에 불변 저장된 이 실험 조건들을 모은다
from harness import result_path
from harness.results import load_result

records = [load_result(result_path(c, step='stepB')) for c in conditions]
print('로드:', len(records), '건 -> results/stepB/')

In [ ]:
# 요약 — 2x2 어텐션(지침 x 그룹), 지침 어텐션 평탄 확인, 층별 궤적
import pandas as pd, matplotlib.pyplot as plt

rows = []
for r in records:
    t = r.condition.instruction.target_notation.value
    for layer, pl in r.metrics.per_layer.items():
        rows.append({'target': t, 'layer': int(layer),
                     'camel_attn': pl.get('code_camel__attention_weight'),
                     'snake_attn': pl.get('code_snake__attention_weight'),
                     'instr_attn': pl.get('instruction__attention_weight'),
                     'camel_av': pl.get('code_camel__av_norm'),
                     'snake_av': pl.get('code_snake__av_norm'),
                     'camel_v': pl.get('code_camel__v_norm'),
                     'snake_v': pl.get('code_snake__v_norm')})
df = pd.DataFrame(rows)
REF_LAYER = int((df.layer.max() + 1) * REF_FRAC)
ref = df[df.layer == REF_LAYER]

# 2x2: 지침(camel/snake) x 어느 그룹 토큰을 더 보는가 @ 기준 층
tab = ref.groupby('target')[['camel_attn', 'snake_attn']].mean()
tab['더_본_쪽'] = np.where(tab['snake_attn'] > tab['camel_attn'], 'snake', 'camel')
print(f'2x2 어텐션 @L{REF_LAYER} (충돌 표기를 더 보는가):'); print(tab.round(4))
print('\n지침 구간 어텐션 (평탄해야 함):')
print(ref.groupby('target')['instr_attn'].mean().round(4))
print('\n‖v‖ (크기, 평탄 기대) @L%d:' % REF_LAYER)
print(ref.groupby('target')[['camel_v', 'snake_v']].mean().round(3))

# 층별 궤적: 지침별 camel/snake 토큰 어텐션
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), sharey=True)
for ax, t in zip(axes, ['camel', 'snake']):
    g = df[df.target == t].groupby('layer')[['camel_attn', 'snake_attn']].mean()
    ax.plot(g.index, g['camel_attn'], marker='.', label='camel 토큰')
    ax.plot(g.index, g['snake_attn'], marker='.', label='snake 토큰')
    ax.axvline(REF_LAYER, color='gray', ls='--', alpha=.5)
    ax.set_title(f'지침={t}'); ax.set_xlabel('layer'); ax.grid(True, alpha=.3); ax.legend()
axes[0].set_ylabel('구간 어텐션 합(평균)')
plt.suptitle('층별 camel vs snake 토큰 어텐션'); plt.tight_layout(); plt.show()

In [ ]:
# 결과 다운로드 — results/stepB 를 zip으로 묶어 내려받는다
import shutil
shutil.make_archive('stepB_results', 'zip', 'results/stepB')
try:
    from google.colab import files
    files.download('stepB_results.zip')
except Exception as e:
    print('Colab 아님(수동 다운로드): stepB_results.zip', e)